# Gold: pump.fun token risk score -- incremental (foreachBatch + MERGE)

Same scoring logic as `pumpapi-lakehouse/transformations/gold_pump_token_risk.py`
(now removed from that Lakeflow pipeline -- a materialized view and a
separately-MERGEd table can't both own `workspace.gold.gold_pump_token_risk`),
restructured to recompute **only** the mints that had new activity since the
last run, instead of the full table every trigger.

Why this can't live inside the Lakeflow declarative pipeline: detecting
"which mints changed" needs a streaming read, but this scoring logic relies
heavily on `Window.partitionBy(...).orderBy(...)` (latest pool state,
first token-creation row, top-holder ranking) -- arbitrary (non-time-based)
window functions aren't supported on streaming DataFrames in Structured
Streaming. `foreachBatch` sidesteps this: inside it, each micro-batch is a
plain *batch* DataFrame, where window functions work normally. This is the
same pattern the dvdrental Silver layer uses for CDC upserts (see
`processing/silver/NB_process_to_silver_generic.ipynb`) -- `apply_changes`/
`dp.table` primitives don't offer this escape hatch, so it has to be a
plain notebook, not part of `pumpapi-lakehouse/`.

Design: `design/pumpfun/RISK_SCORING_DESIGN.md`. Weights/thresholds carried
over unchanged from the Lakeflow version (see that file's git history for
the full investigation trail, including the real rugged-token case this
was tuned against: mint `12DqvhKnLFV9uiGiQYJeAkSLS7eF5FiE5UE61vA1pump`).
`sniper_flip_ratio` was added afterward against a second real case (mint
`12NvSK9hEZsaFzjyNGCAK3mk1UywrL7ENTKJmYKDpump`) that had near-zero holder
concentration and no creator dump -- sniping_ratio alone (20/100 points)
wasn't enough to cross into "medium". A price-drawdown-from-peak signal
was considered and rejected in favor of sniper_flip_ratio: drawdown is
only observable *after* the crash has already happened, while flip ratio
is measurable within ~15-20 minutes of launch, well before that.

`design/pumpfun/RISK_SCORING_GAP_AUDIT.md` then swept 15,912 low-risk
tokens over 30 days for confirmed-but-missed scams and found the dominant
gap wasn't a missing signal at all: `_holder_concentration_signal` was
reconstructed from `silver_pump_transfers` alone, but pump.fun holders
overwhelmingly trade against the bonding curve/pool rather than
transferring peer-to-peer -- across 30 sampled rugged tokens, recorded
concentration read 0.5-16% when the true trade-based concentration was
99.98-99.996%. Fixed by unioning net position from both `pump_trades` and
`pump_transfers` into one ledger. Also adds `scoring_model_version` (see
that constant's comment) to address the audit's third finding: an
incrementally-scored table can silently freeze a dead mint's row at an
older model version's output.

**First run note:** this streaming query has no prior checkpoint, so its
first trigger treats every row currently in `pump_tokens`/`pump_pools`/
`pump_trades`/`pump_transfers` as "new" -- a one-time full backfill touching
every existing mint, comparable in cost to one full run of the old
materialized view. Steady-state runs after that only touch mints with
genuinely new activity.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

TOKENS_TABLE = "workspace.silver.pump_tokens"
POOLS_TABLE = "workspace.silver.pump_pools"
TRADES_TABLE = "workspace.silver.pump_trades"
TRANSFERS_TABLE = "workspace.silver.pump_transfers"
GOLD_TABLE = "workspace.gold.gold_pump_token_risk"
CHECKPOINT_PATH = "/Volumes/workspace/default/mnt/checkpoints/gold_pump_token_risk"

# Bumped whenever the scoring logic changes. The incremental design (see
# the intro cell) only recomputes mints with new source activity, so a
# dead/inactive mint's row silently freezes at whatever an older model
# version last computed -- GAP_AUDIT finding #3
# (design/pumpfun/RISK_SCORING_GAP_AUDIT.md): 28 of 30 audited tokens
# still showed sniper_flip_ratio=NULL despite that signal being live in
# the model, because they'd gone quiet before it shipped. Stamped onto
# every row so a `WHERE scoring_model_version < N` query can find rows
# that need a forced re-score even with no new data. After bumping this,
# run a one-time full re-score (reset CHECKPOINT_PATH, or otherwise force
# every existing mint back through touched_mints_stream) rather than
# relying on organic new activity to eventually reach every row.
SCORING_MODEL_VERSION = 4

# Below this many distinct traders, a mint is flagged has_meaningful_activity
# = false: not "safe", just never traded enough to show (or rule out) any
# manipulation pattern. Real case that prompted this: mint
# C4oBvs4xg31Nr7FpW2ePb1UxE2zgL9ueCoAuwnvtpump had exactly 1 buy + 1 sell,
# ever, market cap flat around $28 the whole time -- risk_score=0 there is
# mathematically correct (no sniping/bundling/concentration/dump signal
# fired because none of that happened), but "no evidence of manipulation"
# and "confirmed safe" are different claims. total_trade_count and
# distinct_trader_count are exposed unconditionally (no baked-in
# threshold) so any consumer -- including the planned Success Score --
# can apply its own bar instead of relying on this one.
MIN_DISTINCT_TRADERS_FOR_MEANINGFUL_ACTIVITY = 3

# How soon after `create` a buy/sell counts as "launch window" activity for
# the bundling and sniping signals.
LAUNCH_WINDOW_MINUTES = 15

# How soon after `create` a burst of distinct buyers counts as sniping.
SNIPE_WINDOW_SECONDS = 60

# Distinct buyers within SNIPE_WINDOW_SECONDS at/above this count is
# treated as maximum sniping risk; below it, risk scales linearly to 0.
SNIPE_COUNT_FOR_MAX_RISK = 50

# How soon after creation a sniper's sell counts as a "flip" for
# sniper_flip_ratio -- confirmed against a real rugged mint
# (12NvSK9hEZsaFzjyNGCAK3mk1UywrL7ENTKJmYKDpump) with low holder
# concentration and no creator dump (so neither of those signals fired),
# yet 258 snipers bought in the first 60s and 251 (97%) sold again within
# 15 minutes -- a much earlier, more specific tell than sheer buyer count
# or the eventual ~1223x price collapse a day later.
FLIP_WINDOW_MINUTES = 15

# sniper_flip_ratio at/above this, with at least MIN_SNIPERS_FOR_FLIP_ESCALATION
# distinct snipers (avoids small-sample noise), floors risk_tier at "high".
EXTREME_SNIPER_FLIP_THRESHOLD = 0.8
MIN_SNIPERS_FOR_FLIP_ESCALATION = 20

# Rolling window (relative to *now*, not token creation) for detecting an
# actively-in-progress creator dump. A dump inside this window escalates
# risk_tier straight to "critical".
CREATOR_DUMP_RECENT_WINDOW_MINUTES = 15

# top_holder_share at/above this floors risk_tier at "high" regardless of
# the weighted score.
EXTREME_CONCENTRATION_THRESHOLD = 0.95

# How many top wallets to sum for the holder-concentration signal.
TOP_HOLDER_COUNT = 10

# burnedLiquidity% at/above this is treated as a fully safe pool (0 risk
# contribution); below it, risk scales linearly down to 0%. pump.fun always
# burns 100% after migration by protocol design, so this signal reads
# "safe" for every pump.fun-native migration regardless of legitimacy --
# it only actually discriminates for Meteora/custom pools.
BURNED_LIQUIDITY_SAFE_PCT = 25.0

# A pool `remove` event is treated as a drain (feeds the `rugged` funnel
# status) when reserves fall below this fraction of their observed peak.
RUG_DRAIN_RATIO = 0.05

# spl-token-2022 extensions that grant the issuer/delegate a way to move,
# block, tax, or freeze a holder's tokens without their consent. See
# design/pumpfun/RISK_SCORING_DESIGN.md for the full source-by-source
# rationale (solana.com/docs/tokens/extensions + offside.io security post).
UNSAFE_TOKEN_EXTENSIONS = [
    "PermanentDelegate",
    "NonTransferable",
    "NonTransferableAccount",
    "Pausable",
    "PausableAccount",
    "DefaultAccountState",
    "TransferHook",
    "TransferHookAccount",
    "TransferFeeConfig",
    "TransferFeeAmount",
    "ConfidentialTransferMint",
    "ConfidentialTransferAccount",
    "ConfidentialTransferFeeConfig",
    "ConfidentialTransferFeeAmount",
    "ConfidentialMintBurn",
]
_UNSAFE_TOKEN_EXTENSION_PATTERN = r"\b(?:" + "|".join(UNSAFE_TOKEN_EXTENSIONS) + r")\b"

# Category B weights (max points each signal can contribute to risk_score).
WEIGHT_BURNED_LIQUIDITY = 35
WEIGHT_LOCKED_LIQUIDITY = 25
WEIGHT_POOL_TRUST = 15
WEIGHT_BUNDLING = 15
WEIGHT_SNIPING = 20
WEIGHT_SNIPER_FLIP = 25
WEIGHT_CREATOR_DUMP_EVER = 30
WEIGHT_HOLDER_CONCENTRATION = 20
WEIGHT_POST_MIGRATION_FEE = 10

In [ ]:
def _clamp01(col):
    return F.greatest(F.lit(0.0), F.least(col, F.lit(1.0)))


def _percent_string_to_double(col):
    """Parse a "NN%"-style string (burnedLiquidity's own glossary example)
    into a 0-100 double. NULL if no numeric prefix is found."""
    digits = F.regexp_extract(col, r"([0-9]+(\.[0-9]+)?)", 1)
    return F.when(digits != "", digits.cast("double"))

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    mint                                    STRING NOT NULL,
    creator_wallet                          STRING,
    created_at                              TIMESTAMP,
    mint_authority_active                   BOOLEAN,
    freeze_authority_active                 BOOLEAN,
    unsafe_token_extension                  BOOLEAN,
    mayhem_mode_on                          BOOLEAN,
    extreme_concentration                   BOOLEAN,
    any_hard_blocker                        BOOLEAN,
    burned_liquidity_pct                    DOUBLE,
    locked_liquidity_after_migration_pct    DOUBLE,
    pool_created_by                         STRING,
    bundling_ratio                          DOUBLE,
    snipe_window_distinct_buyers            BIGINT,
    sniping_ratio                           DOUBLE,
    snipers_flipped_count                   BIGINT,
    sniper_flip_ratio                       DOUBLE,
    extreme_sniper_flip                     BOOLEAN,
    creator_dumped_ever                     BOOLEAN,
    creator_dumped_recent                   BOOLEAN,
    top_holder_share                        DOUBLE,
    pool_fee_rate_after_migration           DOUBLE,
    risk_score                              INT,
    risk_tier                               STRING,
    funnel_status                           STRING,
    total_trade_count                       BIGINT,
    distinct_trader_count                   BIGINT,
    has_meaningful_activity                 BOOLEAN,
    scored_at                               TIMESTAMP,
    scoring_model_version                   INT
) USING DELTA
""")
print(f"{GOLD_TABLE} is ready")

In [ ]:
def _tokens_per_mint(spark, mint_filter):
    """One row per touched mint: creation identity + the two
    hard-blocker fields that live on the `create` event."""
    window = Window.partitionBy("mint").orderBy(F.col("created_at").asc())
    return (
        spark.read.table(TOKENS_TABLE)
        .join(mint_filter, on="mint", how="inner")
        .withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .select(
            "mint",
            "creator_wallet",
            "created_at",
            "mint_authority",
            "freeze_authority",
            "token_program",
            "token_extensions",
        )
    )


def _pool_state_per_mint(spark, mint_filter):
    """Latest known pool state per touched mint, plus two lifecycle
    flags: whether it ever migrated, and whether a `remove` event ever
    drained reserves close to their observed peak (candidate rug signal)."""
    pools = spark.read.table(POOLS_TABLE).join(mint_filter, on="mint", how="inner")
    mint_window = Window.partitionBy("mint")
    latest_window = Window.partitionBy("mint").orderBy(F.col("event_time").desc())

    enriched = (
        pools
        .withColumn("peak_quote_in_pool", F.max("quote_in_pool").over(mint_window))
        .withColumn(
            "has_migrate_event",
            F.max(F.when(F.col("action") == "migrate", 1).otherwise(0)).over(mint_window) == 1,
        )
        .withColumn(
            "is_drain_remove",
            (F.col("action") == "remove")
            & F.col("quote_in_pool").isNotNull()
            & (F.col("peak_quote_in_pool") > 0)
            & (F.col("quote_in_pool") / F.col("peak_quote_in_pool") < F.lit(RUG_DRAIN_RATIO)),
        )
        .withColumn(
            "has_large_remove_event",
            F.max(F.when(F.col("is_drain_remove"), 1).otherwise(0)).over(mint_window) == 1,
        )
    )

    return (
        enriched
        .withColumn("_rn", F.row_number().over(latest_window))
        .filter(F.col("_rn") == 1)
        .withColumn("burned_liquidity_pct", _percent_string_to_double(F.col("burned_liquidity")))
        .withColumn(
            "locked_liquidity_after_migration_pct",
            _percent_string_to_double(F.col("locked_liquidity_after_migration")),
        )
        .select(
            "mint",
            "pool_created_by",
            "burned_liquidity_pct",
            "locked_liquidity_after_migration_pct",
            "pool_fee_rate_after_migration",
            F.coalesce(F.col("mayhem_mode"), F.lit(False)).alias("mayhem_mode_on"),
            F.col("has_migrate_event"),
            F.col("has_large_remove_event"),
        )
    )


def _launch_window_trades(spark, mint_filter, tokens):
    """buy/sell trades within LAUNCH_WINDOW_MINUTES of creation -- the
    shared input for the bundling and sniping signals."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    launch = tokens.select("mint", "created_at")
    return (
        trades.alias("t")
        .join(launch.alias("k"), on="mint", how="inner")
        .filter(
            (F.col("t.event_time") >= F.col("k.created_at"))
            & (
                F.col("t.event_time")
                <= F.col("k.created_at") + F.expr(f"INTERVAL {LAUNCH_WINDOW_MINUTES} MINUTES")
            )
        )
    )


def _bundling_signal(spark, mint_filter, tokens):
    """Share of launch-window buy transactions where breakdown[] shows
    more than one distinct trader in the same signature (atomic bundle)."""
    launch_trades = _launch_window_trades(spark, mint_filter, tokens)
    tx_trader_counts = (
        launch_trades
        .filter(F.col("t.trade_action") == "buy")
        .groupBy("mint", "signature")
        .agg(F.countDistinct("trader").alias("distinct_traders"))
    )
    return (
        tx_trader_counts
        .groupBy("mint")
        .agg(
            F.count("signature").alias("launch_buy_tx_count"),
            F.sum(F.when(F.col("distinct_traders") > 1, 1).otherwise(0)).alias("bundled_tx_count"),
        )
        .withColumn(
            "bundling_ratio",
            F.when(
                F.col("launch_buy_tx_count") > 0,
                F.col("bundled_tx_count") / F.col("launch_buy_tx_count"),
            ).otherwise(F.lit(0.0)),
        )
        .select("mint", "bundling_ratio")
    )


def _sniping_signal(spark, mint_filter, tokens):
    """Distinct buy-traders within SNIPE_WINDOW_SECONDS of creation --
    catches many-separate-transactions bot rushes that _bundling_signal
    structurally can't see (it only counts >1 trader inside one tx)."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    launch = tokens.select("mint", "created_at")
    snipe_window_trades = (
        trades.alias("t")
        .join(launch.alias("k"), on="mint", how="inner")
        .filter(
            (F.col("t.trade_action") == "buy")
            & (F.col("t.event_time") >= F.col("k.created_at"))
            & (
                F.col("t.event_time")
                <= F.col("k.created_at") + F.expr(f"INTERVAL {SNIPE_WINDOW_SECONDS} SECONDS")
            )
        )
    )
    return (
        snipe_window_trades
        .groupBy("mint")
        .agg(F.countDistinct("t.trader").alias("snipe_window_distinct_buyers"))
        .withColumn(
            "sniping_ratio",
            _clamp01(F.col("snipe_window_distinct_buyers") / F.lit(float(SNIPE_COUNT_FOR_MAX_RISK))),
        )
        .select("mint", "snipe_window_distinct_buyers", "sniping_ratio")
    )


def _sniper_flip_signal(spark, mint_filter, tokens):
    """Fraction of launch-window snipers (distinct buyers within
    SNIPE_WINDOW_SECONDS of creation) who sell again within
    FLIP_WINDOW_MINUTES of creation -- a much earlier and more specific
    tell than sheer buyer count. Confirmed against a real rugged mint
    with near-zero holder concentration and no creator dump (neither of
    those signals fired): 258 snipers bought in the first 60s, 251 (97%)
    sold again within 15 minutes -- a textbook buy-and-flip bot pattern,
    available well before the eventual price collapse played out."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    launch = tokens.select("mint", "created_at")

    snipers = (
        trades.alias("t")
        .join(launch.alias("k"), on="mint", how="inner")
        .filter(
            (F.col("t.trade_action") == "buy")
            & (F.col("t.event_time") >= F.col("k.created_at"))
            & (
                F.col("t.event_time")
                <= F.col("k.created_at") + F.expr(f"INTERVAL {SNIPE_WINDOW_SECONDS} SECONDS")
            )
        )
        .select("mint", "trader", F.col("k.created_at").alias("created_at"))
        .distinct()
    )

    sells = (
        trades
        .filter(F.col("trade_action") == "sell")
        .select("mint", "trader", F.col("event_time").alias("sell_time"))
    )

    sniper_sells = (
        snipers.alias("sn")
        .join(sells.alias("se"), on=["mint", "trader"], how="left")
        .withColumn(
            "flipped",
            F.col("se.sell_time").isNotNull()
            & (
                F.col("se.sell_time")
                <= F.col("sn.created_at") + F.expr(f"INTERVAL {FLIP_WINDOW_MINUTES} MINUTES")
            ),
        )
    )

    return (
        sniper_sells
        .groupBy("mint")
        .agg(
            F.countDistinct("trader").alias("snipers_count"),
            F.countDistinct(F.when(F.col("flipped"), F.col("trader"))).alias("snipers_flipped_count"),
        )
        .withColumn(
            "sniper_flip_ratio",
            F.when(
                F.col("snipers_count") > 0,
                F.col("snipers_flipped_count") / F.col("snipers_count"),
            ).otherwise(F.lit(0.0)),
        )
        .select("mint", "snipers_count", "snipers_flipped_count", "sniper_flip_ratio")
    )


def _creator_dump_ever_signal(spark, mint_filter, tokens):
    """Whether the creator wallet ever shows up selling, anywhere in
    the full trade history for this mint (unbounded lookback)."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    creators = tokens.select("mint", "creator_wallet")
    return (
        trades.alias("t")
        .join(creators.alias("k"), on="mint", how="inner")
        .filter(
            (F.col("t.trade_action") == "sell")
            & (F.col("t.trader") == F.col("k.creator_wallet"))
        )
        .select("mint")
        .distinct()
        .withColumn("creator_dumped_ever", F.lit(True))
    )


def _creator_dump_recent_signal(spark, mint_filter, tokens):
    """Whether the creator wallet sold within the last
    CREATOR_DUMP_RECENT_WINDOW_MINUTES, relative to *now*."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    creators = tokens.select("mint", "creator_wallet")
    cutoff = F.current_timestamp() - F.expr(f"INTERVAL {CREATOR_DUMP_RECENT_WINDOW_MINUTES} MINUTES")
    return (
        trades.alias("t")
        .join(creators.alias("k"), on="mint", how="inner")
        .filter(
            (F.col("t.trade_action") == "sell")
            & (F.col("t.trader") == F.col("k.creator_wallet"))
            & (F.col("t.event_time") >= cutoff)
        )
        .select("mint")
        .distinct()
        .withColumn("creator_dumped_recent", F.lit(True))
    )


def _trade_activity_flag(spark, mint_filter):
    return (
        spark.read.table(TRADES_TABLE)
        .join(mint_filter, on="mint", how="inner")
        .select("mint")
        .distinct()
        .withColumn("has_trade_activity", F.lit(True))
    )


def _activity_volume_signal(spark, mint_filter):
    """Raw trading-activity volume per mint: total trade count and
    distinct trader count, exposed with no baked-in threshold so any
    consumer can apply its own bar. has_meaningful_activity is a
    convenience boolean for callers who just want a simple filter --
    see MIN_DISTINCT_TRADERS_FOR_MEANINGFUL_ACTIVITY's comment for why
    this exists as a separate axis from risk_score/risk_tier."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    return (
        trades
        .groupBy("mint")
        .agg(
            F.count(F.lit(1)).alias("total_trade_count"),
            F.countDistinct("trader").alias("distinct_trader_count"),
        )
        .withColumn(
            "has_meaningful_activity",
            F.col("distinct_trader_count") >= F.lit(MIN_DISTINCT_TRADERS_FOR_MEANINGFUL_ACTIVITY),
        )
    )


def _holder_concentration_signal(spark, mint_filter):
    """Share of net-positive token balance held by the top N wallets.

    GAP_AUDIT finding (design/pumpfun/RISK_SCORING_GAP_AUDIT.md, dominant
    finding): this used to be reconstructed from silver_pump_transfers
    alone, but on pump.fun holders overwhelmingly acquire/dispose of
    tokens by trading against the bonding curve/pool (buy/sell in
    pump_trades), not by peer-to-peer wallet transfers -- pump_transfers
    only captures a minority side-channel (withdrawals, wallet
    consolidation). Across 30 audited rugged tokens, transfers-only
    concentration read 0.5-16% while the true trade-based concentration
    was 99.98-99.996% -- a 40x-4000x understatement in the wrong
    direction, silently defeating the extreme_concentration escalator.
    Now unions net position from BOTH pump_trades (buy=+, sell=-) and
    pump_transfers (to=+, from=-) into one ledger per (mint, wallet) --
    strictly more complete than either source alone."""
    trades = spark.read.table(TRADES_TABLE).join(mint_filter, on="mint", how="inner")
    transfers = spark.read.table(TRANSFERS_TABLE).join(mint_filter, on="mint", how="inner")

    trade_buys = (
        trades.filter(F.col("trade_action") == "buy")
        .select("mint", F.col("trader").alias("wallet"), F.col("token_amount").alias("amount"))
    )
    trade_sells = (
        trades.filter(F.col("trade_action") == "sell")
        .select("mint", F.col("trader").alias("wallet"), (-F.col("token_amount")).alias("amount"))
    )
    transfer_in = transfers.select("mint", F.col("to_wallet").alias("wallet"), F.col("amount"))
    transfer_out = transfers.select(
        "mint", F.col("from_wallet").alias("wallet"), (-F.col("amount")).alias("amount")
    )

    net_balances = (
        trade_buys
        .unionByName(trade_sells)
        .unionByName(transfer_in)
        .unionByName(transfer_out)
        .groupBy("mint", "wallet")
        .agg(F.sum("amount").alias("net_amount"))
        .filter(F.col("net_amount") > 0)
    )

    rank_window = Window.partitionBy("mint").orderBy(F.col("net_amount").desc())
    ranked = net_balances.withColumn("_rank", F.row_number().over(rank_window))

    totals = net_balances.groupBy("mint").agg(F.sum("net_amount").alias("total_positive_balance"))
    top_holders = (
        ranked
        .filter(F.col("_rank") <= TOP_HOLDER_COUNT)
        .groupBy("mint")
        .agg(F.sum("net_amount").alias("top_holder_balance"))
    )

    return (
        totals
        .join(top_holders, on="mint", how="left")
        .withColumn(
            "top_holder_share",
            F.when(
                F.col("total_positive_balance") > 0,
                F.coalesce(F.col("top_holder_balance"), F.lit(0.0)) / F.col("total_positive_balance"),
            ),
        )
        .select("mint", "top_holder_share")
    )

In [ ]:
def compute_risk_for_mints(spark, mint_filter):
    """Full risk-scoring logic, computed only for the mints present in
    mint_filter (a DataFrame with a single distinct `mint` column). Ported
    unchanged from pumpapi-lakehouse/transformations/gold_pump_token_risk.py
    -- see design/pumpfun/RISK_SCORING_DESIGN.md for the scoring rationale."""
    tokens = _tokens_per_mint(spark, mint_filter)
    pools = _pool_state_per_mint(spark, mint_filter)
    bundling = _bundling_signal(spark, mint_filter, tokens)
    sniping = _sniping_signal(spark, mint_filter, tokens)
    sniper_flip = _sniper_flip_signal(spark, mint_filter, tokens)
    creator_dump_ever = _creator_dump_ever_signal(spark, mint_filter, tokens)
    creator_dump_recent = _creator_dump_recent_signal(spark, mint_filter, tokens)
    trade_activity = _trade_activity_flag(spark, mint_filter)
    activity_volume = _activity_volume_signal(spark, mint_filter)
    holders = _holder_concentration_signal(spark, mint_filter)

    joined = (
        tokens
        .join(pools, on="mint", how="left")
        .join(bundling, on="mint", how="left")
        .join(sniping, on="mint", how="left")
        .join(sniper_flip, on="mint", how="left")
        .join(creator_dump_ever, on="mint", how="left")
        .join(creator_dump_recent, on="mint", how="left")
        .join(trade_activity, on="mint", how="left")
        .join(activity_volume, on="mint", how="left")
        .join(holders, on="mint", how="left")
        .withColumn("mayhem_mode_on", F.coalesce(F.col("mayhem_mode_on"), F.lit(False)))
        .withColumn("has_migrate_event", F.coalesce(F.col("has_migrate_event"), F.lit(False)))
        .withColumn("has_large_remove_event", F.coalesce(F.col("has_large_remove_event"), F.lit(False)))
        .withColumn("bundling_ratio", F.coalesce(F.col("bundling_ratio"), F.lit(0.0)))
        .withColumn("sniping_ratio", F.coalesce(F.col("sniping_ratio"), F.lit(0.0)))
        .withColumn("snipe_window_distinct_buyers", F.coalesce(F.col("snipe_window_distinct_buyers"), F.lit(0)))
        .withColumn("snipers_count", F.coalesce(F.col("snipers_count"), F.lit(0)))
        .withColumn("snipers_flipped_count", F.coalesce(F.col("snipers_flipped_count"), F.lit(0)))
        .withColumn("sniper_flip_ratio", F.coalesce(F.col("sniper_flip_ratio"), F.lit(0.0)))
        .withColumn("creator_dumped_ever", F.coalesce(F.col("creator_dumped_ever"), F.lit(False)))
        .withColumn("creator_dumped_recent", F.coalesce(F.col("creator_dumped_recent"), F.lit(False)))
        .withColumn("has_trade_activity", F.coalesce(F.col("has_trade_activity"), F.lit(False)))
        .withColumn("total_trade_count", F.coalesce(F.col("total_trade_count"), F.lit(0)))
        .withColumn("distinct_trader_count", F.coalesce(F.col("distinct_trader_count"), F.lit(0)))
        .withColumn("has_meaningful_activity", F.coalesce(F.col("has_meaningful_activity"), F.lit(False)))
    )

    # --- Category A: hard blockers (boolean, not weighted) ---
    with_flags = (
        joined
        .withColumn("mint_authority_active", F.col("mint_authority").isNotNull())
        .withColumn("freeze_authority_active", F.col("freeze_authority").isNotNull())
        .withColumn(
            "unsafe_token_extension",
            F.col("token_extensions").isNotNull()
            & F.col("token_extensions").rlike(_UNSAFE_TOKEN_EXTENSION_PATTERN),
        )
        .withColumn(
            "extreme_concentration",
            F.col("top_holder_share").isNotNull()
            & (F.col("top_holder_share") >= F.lit(EXTREME_CONCENTRATION_THRESHOLD)),
        )
        .withColumn(
            "extreme_sniper_flip",
            (F.col("snipers_count") >= F.lit(MIN_SNIPERS_FOR_FLIP_ESCALATION))
            & (F.col("sniper_flip_ratio") >= F.lit(EXTREME_SNIPER_FLIP_THRESHOLD)),
        )
    )

    # --- Category B: weighted score (0-100) ---
    pool_trust_risk = (
        F.when(F.col("pool_created_by") == "custom", F.lit(1.0))
        .when(
            (F.col("pool_created_by") == "meteora-launchpad")
            & (
                F.col("locked_liquidity_after_migration_pct").isNull()
                | (F.col("locked_liquidity_after_migration_pct") < 100)
            ),
            F.lit(1.0),
        )
        .otherwise(F.lit(0.0))
    )

    with_score = (
        with_flags
        .withColumn(
            "score_burned_liquidity",
            F.when(
                F.col("burned_liquidity_pct").isNotNull(),
                F.lit(WEIGHT_BURNED_LIQUIDITY)
                * _clamp01(F.lit(1.0) - F.col("burned_liquidity_pct") / F.lit(BURNED_LIQUIDITY_SAFE_PCT)),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "score_locked_liquidity",
            F.when(
                F.col("locked_liquidity_after_migration_pct").isNotNull(),
                F.lit(WEIGHT_LOCKED_LIQUIDITY)
                * _clamp01(F.lit(1.0) - F.col("locked_liquidity_after_migration_pct") / F.lit(100.0)),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn("score_pool_trust", F.lit(WEIGHT_POOL_TRUST) * pool_trust_risk)
        .withColumn("score_bundling", F.lit(WEIGHT_BUNDLING) * _clamp01(F.col("bundling_ratio")))
        .withColumn("score_sniping", F.lit(WEIGHT_SNIPING) * _clamp01(F.col("sniping_ratio")))
        .withColumn("score_sniper_flip", F.lit(WEIGHT_SNIPER_FLIP) * _clamp01(F.col("sniper_flip_ratio")))
        .withColumn(
            "score_creator_dump",
            F.when(F.col("creator_dumped_ever"), F.lit(float(WEIGHT_CREATOR_DUMP_EVER))).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "score_holder_concentration",
            F.when(
                F.col("top_holder_share").isNotNull(),
                F.lit(WEIGHT_HOLDER_CONCENTRATION) * _clamp01(F.col("top_holder_share")),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "score_post_migration_fee",
            F.when(
                F.col("pool_fee_rate_after_migration").isNotNull(),
                F.lit(WEIGHT_POST_MIGRATION_FEE) * _clamp01(F.col("pool_fee_rate_after_migration") / F.lit(0.1)),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "risk_score",
            F.least(
                F.round(
                    F.col("score_burned_liquidity")
                    + F.col("score_locked_liquidity")
                    + F.col("score_pool_trust")
                    + F.col("score_bundling")
                    + F.col("score_sniping")
                    + F.col("score_sniper_flip")
                    + F.col("score_creator_dump")
                    + F.col("score_holder_concentration")
                    + F.col("score_post_migration_fee")
                ),
                F.lit(100.0),
            ).cast("int"),
        )
    )

    # --- Tier: score buckets, floored by escalators ---
    with_tier = (
        with_score
        .withColumn(
            "score_tier",
            F.when(F.col("risk_score") >= 81, F.lit("critical"))
            .when(F.col("risk_score") >= 51, F.lit("high"))
            .when(F.col("risk_score") >= 21, F.lit("medium"))
            .otherwise(F.lit("low")),
        )
        .withColumn(
            "any_hard_blocker",
            F.col("mint_authority_active")
            | F.col("freeze_authority_active")
            | F.col("unsafe_token_extension")
            | F.col("mayhem_mode_on"),
        )
        .withColumn(
            "risk_tier",
            F.when(F.col("creator_dumped_recent"), F.lit("critical"))
            .when(
                (F.col("any_hard_blocker") | F.col("extreme_concentration") | F.col("extreme_sniper_flip"))
                & F.col("score_tier").isin("low", "medium"),
                F.lit("high"),
            )
            .otherwise(F.col("score_tier")),
        )
    )

    # --- Category C: migration funnel status ---
    with_funnel = with_tier.withColumn(
        "funnel_status",
        F.when(
            F.col("has_large_remove_event")
            & (F.coalesce(F.col("burned_liquidity_pct"), F.lit(0.0)) < F.lit(BURNED_LIQUIDITY_SAFE_PCT)),
            F.lit("rugged"),
        )
        .when(F.col("has_migrate_event"), F.lit("migrated"))
        .when(F.col("has_trade_activity"), F.lit("active"))
        .otherwise(F.lit("created")),
    )

    return with_funnel.select(
        "mint",
        "creator_wallet",
        "created_at",
        "mint_authority_active",
        "freeze_authority_active",
        "unsafe_token_extension",
        "mayhem_mode_on",
        "extreme_concentration",
        "any_hard_blocker",
        "burned_liquidity_pct",
        "locked_liquidity_after_migration_pct",
        "pool_created_by",
        "bundling_ratio",
        "snipe_window_distinct_buyers",
        "sniping_ratio",
        "snipers_flipped_count",
        "sniper_flip_ratio",
        "extreme_sniper_flip",
        "creator_dumped_ever",
        "creator_dumped_recent",
        "top_holder_share",
        "pool_fee_rate_after_migration",
        "risk_score",
        "risk_tier",
        "funnel_status",
        "total_trade_count",
        "distinct_trader_count",
        "has_meaningful_activity",
        F.current_timestamp().alias("scored_at"),
        F.lit(SCORING_MODEL_VERSION).alias("scoring_model_version"),
    )

In [ ]:
def upsert_gold_risk(batch_df, batch_id):
    if not batch_df.take(1):
        return

    # Spark Connect: must use batch_df.sparkSession, not the global spark --
    # matches the convention in NB_process_to_silver_generic.ipynb.
    _spark = batch_df.sparkSession

    touched_mints = batch_df.select("mint").distinct()
    touched_count = touched_mints.count()
    result = compute_risk_for_mints(_spark, touched_mints)

    delta_table = DeltaTable.forName(_spark, GOLD_TABLE)
    (
        delta_table.alias("t")
        .merge(result.alias("s"), "t.mint = s.mint")
        # .withSchemaEvolution() (not the session-level
        # spark.databricks.delta.schema.autoMerge.enabled conf, which
        # serverless rejects: "This setting is not supported in
        # serverless environments") lets new signal columns -- added to
        # `result`'s schema after the target table was first created --
        # merge in automatically instead of failing schema checks.
        .withSchemaEvolution()
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"batch {batch_id}: recomputed risk for {touched_count} touched mint(s)")

In [ ]:
# Streaming union of `mint` from all 4 source Silver tables -- each
# micro-batch surfaces exactly the mints with new activity since the last
# successful run (Lakeflow's/Spark's own streaming checkpoint tracks this
# for free), driving upsert_gold_risk's selective, not full-table,
# recomputation above.
touched_mints_stream = (
    spark.readStream.table(TOKENS_TABLE).select("mint")
    .unionByName(spark.readStream.table(POOLS_TABLE).select("mint"))
    .unionByName(spark.readStream.table(TRADES_TABLE).select("mint"))
    .unionByName(spark.readStream.table(TRANSFERS_TABLE).select("mint"))
)

query = (
    touched_mints_stream.writeStream
    .foreachBatch(upsert_gold_risk)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
print("gold_pump_token_risk incremental run complete.")